# Free Colab recovery probe (issue #39)

Thin wrapper around the checked-in CLI — no trainer logic here. Run at most **5 optimizer updates per session**, export + download the resume bundle, and on the next session upload + import it and continue. `--max-optimizer-updates 5` means *five additional* updates; the checkpoint retains the cumulative count and a resumed run never repeats or skips an update. Never overwrite the preserved source checkpoint. Label every candidate `promoted_*` or `rejected_*`.

In [ ]:
# Cell 1 — confirm the assigned GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU assigned; Runtime > Change runtime type > T4 GPU, then reconnect."
p = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0), "| VRAM GiB:", round(p.total_memory / 2**30, 2))

In [ ]:
# Cell 2 — clone + install (CPU-heavy install first; enable GPU only for training cells)
import os
if not os.path.exists("research-paper"):
    !git clone https://github.com/iamdarshg/research-paper.git
%cd /content/research-paper
!python -m pip install --upgrade pip -q
!pip install -r CLI/requirements.txt -q
!pip install -r requirements-dev.txt -q

In [ ]:
# Cell 3 — CPU preflight before spending GPU quota
!python CLI/aircraft_diffusion_cfd.py info
!python -m pytest -q tests/test_issue39_resume_interface.py
print("Preflight OK — proceed only if all tests pass.")

In [ ]:
# Cell 4 — upload the resume bundle from the previous session (skip on session 0)
# After uploading: the bundle must verify checksums before anything is extracted.
from google.colab import files
uploaded = files.upload()
bundle = next(iter(uploaded))
!python CLI/import_resume_bundle.py --input "/content/{bundle}" --output /content/output/session_NNN --print-resume-command

In [ ]:
# Cell 5 — run one bounded session: 5 ADDITIONAL optimizer updates
# Session 0: drop --resume-from and use --max-optimizer-updates 1 (one-update validation).
MANIFEST = "docs/dataset/minimal_grounded_manifest.jsonl"   # override for the full corpus manifest
RESUME = ""          # e.g. "/content/output/session_NNN/checkpoint_updates_000005.pt"
SAVE_DIR = "/content/output/session_NNN"
!python CLI/run_monitored_training.py \
  --manifest {MANIFEST} \
  --batch-size 1 --grid-size 96 --solver D3Q27 \
  --direct-solver-steps 50 --direct-solver-directions 16 \
  {("--resume-from " + RESUME) if RESUME else ""} \
  --max-optimizer-updates 5 --checkpoint-every-updates 1 \
  --fixed-validation-seeds 0,1,2,3,4,5 \
  --save-dir {SAVE_DIR} --cpu-threads 4

In [ ]:
# Cell 6 — export the resume bundle and download it BEFORE the session can die
!python CLI/export_resume_bundle.py --input {SAVE_DIR} --output /content/resume_session_NNN.tar.gz
from google.colab import files
files.download("/content/resume_session_NNN.tar.gz")
print("Bundle downloaded — verify its checksum locally before the next session.")

## Promotion discipline
- After every 5-update chunk, evaluate on the same six fixed seeds and label candidates `promoted_*` / `rejected_*` with a report.
- Stop immediately on: nonfinite values, solver-coverage failure, occupancy collapse, missing lineage, or a material worst-seed validity regression.
- Promote only if the candidate improves the lexicographic validation rank and passes all issue-#39 mandatory checks.
- A notebook disconnect is not a scientific failure; losing an unexported checkpoint is an infrastructure failure.